# 🌾 Seasonal Agriculture Performance Analysis

**Major Project — Data Visualization using Python**

This project analyzes agricultural performance across seasons, regions, environmental conditions, resource usage, and economic outcomes. The supplied dataset contains **4,000 records and 28 variables**.

## Problem Statement

Agricultural performance is influenced by seasonal environmental conditions, farming practices, resource availability, and market conditions. This project uses data analytics and visualization to identify meaningful seasonal patterns and support evidence-based agricultural planning.

## Analytical Questions
1. How does yield vary across seasons?
2. Which season shows stronger economic performance?
3. How are environmental conditions associated with yield?
4. How does irrigation method relate to water efficiency?
5. Is fertilizer usage associated with yield?
6. Which regions and crops show notable performance differences?
7. What unusual patterns deserve further investigation?

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid')
print('Libraries imported successfully!')

## Load Dataset

The loader checks common Colab and repository-relative locations so the notebook is easier to reuse.

In [ ]:
possible_paths = [
    '/content/seasonal_agriculture_performance_dataset.csv',
    'data/seasonal_agriculture_performance_dataset.csv',
    '../data/seasonal_agriculture_performance_dataset.csv'
]
DATA_PATH = next((p for p in possible_paths if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError('Dataset not found. Upload seasonal_agriculture_performance_dataset.csv to Colab or place it in the repository data folder.')
df = pd.read_csv(DATA_PATH)
print('Dataset loaded successfully!')
print('Using:', DATA_PATH)
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
display(df.head())

## Dataset Overview and Data Quality

In [ ]:
display(df.info())
display(df.describe(include='all').T)
print('Duplicate rows:', df.duplicated().sum())
display(df.isnull().sum().sort_values(ascending=False).head(15))

In [ ]:
clean_df = df.copy()
clean_df.columns = (clean_df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace(r'[^a-z0-9_]', '', regex=True))
print(clean_df.columns.tolist())

## Seasonal Distribution

In [ ]:
display(clean_df['season'].value_counts())
plt.figure(figsize=(8,5))
sns.countplot(data=clean_df, x='season')
plt.title('Distribution of Records Across Seasons')
plt.tight_layout(); plt.show()

## Yield Performance by Season

In [ ]:
season_yield = clean_df.groupby('season')['yield_tonnes_ha'].agg(['mean','median','std']).sort_values('mean', ascending=False)
display(season_yield)
plt.figure(figsize=(8,5))
sns.barplot(data=clean_df, x='season', y='yield_tonnes_ha', errorbar=None)
plt.title('Average Yield by Season')
plt.tight_layout(); plt.show()
plt.figure(figsize=(8,5))
sns.boxplot(data=clean_df, x='season', y='yield_tonnes_ha')
plt.title('Yield Distribution Across Seasons')
plt.tight_layout(); plt.show()

## Economic Performance

In [ ]:
economic = clean_df.groupby('season')[['revenue_inr','total_cost_inr','profit_inr']].mean().sort_values('profit_inr', ascending=False)
display(economic)
plt.figure(figsize=(8,5))
sns.barplot(data=clean_df, x='season', y='profit_inr', errorbar=None)
plt.title('Average Profit by Season')
plt.tight_layout(); plt.show()

## Environmental Conditions vs Yield

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data=clean_df, x='rainfall_mm', y='yield_tonnes_ha', hue='season', alpha=0.65)
plt.title('Rainfall vs Yield')
plt.tight_layout(); plt.show()
plt.figure(figsize=(8,6))
sns.scatterplot(data=clean_df, x='avg_temperature_c', y='yield_tonnes_ha', hue='season', alpha=0.65)
plt.title('Temperature vs Yield')
plt.tight_layout(); plt.show()

## Irrigation and Water Efficiency

In [ ]:
irrigation = clean_df.groupby('irrigation_method')['water_efficiency_t_per_1000m3'].agg(['mean','median']).sort_values('mean', ascending=False)
display(irrigation)
plt.figure(figsize=(9,5))
sns.barplot(data=clean_df, x='irrigation_method', y='water_efficiency_t_per_1000m3', errorbar=None)
plt.title('Water Efficiency by Irrigation Method')
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

## Fertilizer Usage vs Yield

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data=clean_df, x='fertilizer_kg_ha', y='yield_tonnes_ha', hue='season', alpha=0.65)
plt.title('Fertilizer Usage vs Yield')
plt.tight_layout(); plt.show()

## Correlation Analysis

In [ ]:
corr_cols = ['rainfall_mm','avg_temperature_c','humidity_pct','soil_moisture_pct','nitrogen_kg_ha','phosphorus_kg_ha','potassium_kg_ha','fertilizer_kg_ha','yield_tonnes_ha','profit_inr','water_efficiency_t_per_1000m3']
corr = clean_df[corr_cols].corr()
display(corr['yield_tonnes_ha'].sort_values(ascending=False).to_frame('correlation_with_yield'))
plt.figure(figsize=(11,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout(); plt.show()

## Regional Profitability

In [ ]:
state_profit = clean_df.groupby('state')['profit_inr'].mean().sort_values(ascending=False)
display(state_profit.head(10))
top_states = state_profit.head(10)
plt.figure(figsize=(9,5))
sns.barplot(x=top_states.values, y=top_states.index)
plt.title('Top 10 States by Average Profit')
plt.tight_layout(); plt.show()

## Statistical Analysis — ANOVA

In [ ]:
groups = [clean_df.loc[clean_df['season'] == s, 'yield_tonnes_ha'].dropna() for s in clean_df['season'].dropna().unique()]
f_stat, p_value = f_oneway(*groups)
print('ANOVA F-statistic:', f_stat)
print('ANOVA p-value:', p_value)
print('Interpretation: p < 0.05 suggests at least one season has a statistically different mean yield.')

## Exceptional Yield / Outlier Analysis

In [ ]:
Q1 = clean_df['yield_tonnes_ha'].quantile(0.25)
Q3 = clean_df['yield_tonnes_ha'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR
outliers = clean_df[clean_df['yield_tonnes_ha'] > upper_limit]
print('Upper outlier limit:', upper_limit)
print('Exceptional-yield records:', len(outliers))
display(outliers.head(10))

## Does Higher Yield Mean Higher Profit?

In [ ]:
yield_profit_corr = clean_df['yield_tonnes_ha'].corr(clean_df['profit_inr'])
print('Yield-profit correlation:', yield_profit_corr)
plt.figure(figsize=(8,6))
sns.scatterplot(data=clean_df, x='yield_tonnes_ha', y='profit_inr', hue='season', alpha=0.65)
plt.title('Yield vs Profit')
plt.tight_layout(); plt.show()

## Crop × Season Performance

In [ ]:
crop_season = clean_df.pivot_table(values='yield_tonnes_ha', index='crop', columns='season', aggfunc='mean')
display(crop_season)
plt.figure(figsize=(10,8))
sns.heatmap(crop_season, annot=True, fmt='.2f', cmap='YlGnBu')
plt.title('Average Yield by Crop and Season')
plt.xlabel('Season'); plt.ylabel('Crop')
plt.tight_layout(); plt.show()

# 🎯 Final Project Insights

- Kharif has the highest observed average yield and average profit among the three seasons in the supplied analysis.
- Zaid shows negative average profit and deserves closer economic investigation.
- Rainfed farming shows the highest average water-efficiency measure in this dataset.
- Simple correlations should be interpreted as associations, not proof of causation.
- Exceptional-yield records should be investigated rather than automatically deleted.
- Crop × season combinations provide a useful basis for targeted seasonal planning.

## Data-Driven Recommendations
- Use seasonal performance as one input for crop planning rather than relying on season alone.
- Investigate the causes of negative average profitability in Zaid.
- Consider water efficiency when evaluating irrigation strategies.
- Optimize fertilizer and rainfall-related decisions using crop- and region-specific evidence.
- Investigate exceptional-yield farms for potentially repeatable practices.
- Combine yield, market price, and production cost when making profitability decisions.

## Conclusion
The project combines data cleaning, exploratory analysis, statistical testing, and visualization to understand seasonal agricultural performance and support evidence-based agricultural planning.